<a href="https://colab.research.google.com/github/Sadifs/text-analytics-a3-jamjoom-sarbazi/blob/main/05_interpretation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Topic Interpretation (Student A)
**BSAN 6200 Assignment 3 | Disneyland Reviews**

Student A interprets Topics 1–5 for all three parks.

Process followed for each topic:
1. Read top 20 words from model output
2. Read 5–8 representative documents (highest topic-probability reviews)
3. Identified recurring patterns and sentiment
4. Drafted topic name independently (no AI used)
5. Reviewed with partner — notes at end of notebook

## Setup — Load Models

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv('data/processed/disneyland_reviews_processed.csv')
target_groups  = ['California', 'Paris', 'Hong Kong']
selected_k_lda = {'California': 10, 'Paris': 10, 'Hong Kong': 10}

def topic_word_table(components, feature_names, top_n=20):
    rows = []
    for idx, topic in enumerate(components):
        top_ids   = topic.argsort()[-top_n:][::-1]
        top_words = [feature_names[i] for i in top_ids]
        rows.append({'topic_id': idx+1, 'top_words': ', '.join(top_words)})
    return pd.DataFrame(rows)

def top_docs_for_each_topic(subset_df, doc_topic_matrix, top_n_docs=8):
    rows = []
    for t in range(doc_topic_matrix.shape[1]):
        probs = doc_topic_matrix[:, t]
        for rank, doc_id in enumerate(np.argsort(probs)[::-1][:top_n_docs], 1):
            rows.append({'topic_id': t+1, 'rank': rank,
                         'probability': float(probs[doc_id]),
                         'text': subset_df.iloc[doc_id]['review_text'][:800]})
    return pd.DataFrame(rows)

def topic_prevalence_table(doc_topic_matrix):
    avg_probs = doc_topic_matrix.mean(axis=0)
    return pd.DataFrame({'topic_id': range(1, len(avg_probs)+1),
                         'avg_topic_probability': avg_probs})

lda_topic_word_tables = {}
lda_top_doc_tables    = {}
lda_prevalence_tables = {}

for group in target_groups:
    k    = selected_k_lda[group]
    safe = group.lower().replace(' ', '_')
    with open(f'models/lda_{safe}_k{k}.pkl', 'rb') as f:
        fitted = pickle.load(f)
    subset = df[df['location_clean'] == group].copy().reset_index(drop=True)
    lda_topic_word_tables[group] = topic_word_table(fitted['model'].components_, fitted['feature_names'])
    lda_top_doc_tables[group]    = top_docs_for_each_topic(subset, fitted['doc_topic'])
    lda_prevalence_tables[group] = topic_prevalence_table(fitted['doc_topic'])
    print(f'{group}: loaded k={k} model ✓')

California: loaded k=10 model ✓
Paris: loaded k=10 model ✓
Hong Kong: loaded k=10 model ✓


## Reference: Top 20 Words Per Topic

In [ ]:
for group in target_groups:
    print(f'\n{"="*60}')
    print(f'  {group} — top 20 words per topic')
    print(f'{"="*60}')
    display(lda_topic_word_tables[group])


  California — top 20 words per topic


,topic_id,top_words
0,1,"hotel, take, water, around, parade, bring, kid..."
1,2,"food, restaurant, character, experience, eat, ..."
2,3,"would, year, price, many, crowd, money, ticket..."
3,4,"parade, firework, show, night, amazing, fun, w..."
4,5,"year, kid, old, year old, loved, love, every, ..."
5,6,"always, earth, happiest, fun, happiest earth, ..."
6,7,"wait, line, minute, long, hour, early, hallowe..."
7,8,"fast, pas, fast pas, pass, fast pass, use, wai..."
8,9,"world, member, cast, cast member, florida, vis..."
9,10,"mountain, space, space mountain, pirate, jones..."



  Paris — top 20 words per topic


,topic_id,top_words
0,1,"ticket, pas, paris, fast, studio, fast pas, tr..."
1,2,"food, queue, euro, expensive, drink, fast, hou..."
2,3,"staff, would, closed, member, paris, year, cou..."
3,4,"attraction, child, kid, queue, minute, waiting..."
4,5,"paris, florida, world, orlando, magic, attract..."
5,6,"year, amazing, parade, show, magical, loved, k..."
6,7,"mountain, thunder, space, thunder mountain, sp..."
7,8,"character, queue, parade, child, princess, mee..."
8,9,"hotel, bus, train, station, min, shuttle, vill..."
9,10,"hotel, restaurant, stayed, would, show, room, ..."



  Hong Kong — top 20 words per topic


,topic_id,top_words
0,1,"hong, kong, hong kong, smaller, small, visit, ..."
1,2,"food, water, expensive, bring, drink, inside, ..."
2,3,"staff, would, food, queue, small, well, attrac..."
3,4,"queue, ticket, photo, take, pas, character, ar..."
4,5,"kid, child, visit, adult, young, family, small..."
5,6,"hotel, wait, would, recommend, stayed, min, lo..."
6,7,"year, old, year old, daughter, kid, visit, lov..."
7,8,"train, mtr, ticket, easy, station, sunny, bay,..."
8,9,"show, parade, firework, night, king, lion, lio..."
9,10,"mountain, space, land, space mountain, mystic,..."


## Reference: Representative Documents (Topics 1–5)

In [ ]:
for group in target_groups:
    print(f'\n{"="*60}')
    print(f'  {group} — representative docs for Topics 1–5')
    print(f'{"="*60}')
    docs = lda_top_doc_tables[group][lda_top_doc_tables[group]['topic_id'] <= 5]
    for _, row in docs.iterrows():
        print(f'\n--- Topic {row["topic_id"]} | Rank {row["rank"]} | prob={row["probability"]:.3f} ---')
        print(row['text'])


  California — representative docs for Topics 1–5

--- Topic 1 | Rank 1 | prob=0.990 ---
Listen we did our research this time around. We really couldn't afford to buy food at Disneyland, so we found out that they had lockers at the entrance and we used it! Sooo much better if your trying to save your pennies! We brought sandwiches In a small cooler with bags if chips and waters. I was really nervous that the cooler wouldn't fit, but we got the large locker and we still had room to spare! Once lunch time arrived we got our hand stamped, got our food and almost decided to sit on the ground and eat by the entrance. Then we noticed there were were tables behind a locked gate. We walked around the gate towards the spot where the trams drop off and pick up in downtown district. If your not lookin, you'll miss it. But we found a hidden rest area filled with tables, water fountains an

--- Topic 1 | Rank 2 | prob=0.989 ---
Disney   Quality. And nobody does it better. Some tips:  Measure the h

## Reference: Topic Prevalence

In [ ]:
for group in target_groups:
    print(f'\n{"="*60}')
    print(f'  {group} — topic prevalence')
    print(f'{"="*60}')
    display(lda_prevalence_tables[group].sort_values('avg_topic_probability', ascending=False))


  California — topic prevalence


,topic_id,avg_topic_probability
4,5,0.121197
2,3,0.117838
8,9,0.112542
5,6,0.112037
3,4,0.098467
0,1,0.097566
1,2,0.094395
7,8,0.091987
6,7,0.085902
9,10,0.068070



  Paris — topic prevalence


,topic_id,avg_topic_probability
5,6,0.158887
7,8,0.138434
1,2,0.122057
2,3,0.115604
3,4,0.102303
4,5,0.089536
0,1,0.085000
6,7,0.075291
9,10,0.073634
8,9,0.039255



  Hong Kong — topic prevalence


,topic_id,avg_topic_probability
2,3,0.144349
4,5,0.144102
0,1,0.128671
8,9,0.127692
3,4,0.096588
6,7,0.093628
9,10,0.085721
1,2,0.072769
5,6,0.053765
7,8,0.052716


## Topic Interpretations — Student A (Topics 1–5)

### California



**Topic 1 — Practical Visit Tips & Logistics Planning** *(Prevalence: 9.8%)*

Top words: `hotel, take, water, around, parade, bring, kid, early, bus, locker`

Reviews in this topic are written as visitor guides — experienced visitors sharing specific, actionable advice for navigating the park. Recurring themes include bringing your own food and using lockers at the entrance to save money, arriving early to beat queues, and using Fastpass for popular rides. Sentiment is mixed but informative rather than strongly positive or negative. In the representative documents, reviewers specifically mention using lockers to bring sandwiches and a cooler, finding a hidden rest area near the tram drop-off with tables and water fountains, arriving 15 minutes before rides open, and bringing bandaids and blister protection for the long walking day.


**Topic 2 — Food Quality, Dining Venues & Culinary Highlights** *(Prevalence: 9.4%)*

Top words: `food, restaurant, character, experience, eat, blue, bayou, main, street, tour`

This topic clusters reviews centred on specific dining experiences inside the park — named restaurants, particular dishes, and food quality comparisons. Sentiment is mixed: some reviewers celebrate standout dishes and venues, while others mourn discontinued menu items or criticise declining quality. In the representative documents, reviewers specifically name Blue Bayou (praising the Monte Cristo sandwich and plantation veranda setting), Cafe Orleans, Plaza Inn fried chicken on Main Street, Trader Sam's Tiki Bar near the original Disneyland Hotel, and the Napa Rose inside the hotel as first-rate. One reviewer lists over a dozen discontinued dishes by name across multiple restaurants.


**Topic 3 — Overcrowding, Ticket Prices & Value for Money** *(Prevalence: 11.8%)*

Top words: `would, year, price, many, crowd, money, ticket, cost, long, line`

This topic captures visitor frustration with the gap between what Disneyland costs and what the experience delivers when the park is at capacity. Annual passholders feature heavily, comparing current crowded conditions to better past visits. Sentiment is predominantly negative. In the representative documents, reviewers describe travelling from Australia only to find the experience a deep disappointment, paying for premium upgrades that made no difference to wait times, simultaneously finding 3–4 major attractions closed for maintenance, and seriously considering not renewing annual passes after particularly crowded visits.


**Topic 4 — Night Shows, Fireworks & Parade Spectacles** *(Prevalence: 9.8%)*

Top words: `parade, firework, show, night, amazing, fun, watch, fantasmic, paint, world`

Reviews in this topic focus on evening entertainment — the Paint the Night parade, Fantasmic water show, and fireworks display. First-time visitors dominate and use superlative language throughout. Sentiment is overwhelmingly positive. In the representative documents, reviewers describe the Paint the Night parade floats as mind-blowing with extravagant illuminated characters, the fireworks as breathtaking with synchronised movie clips, and Fantasmic as spectacular. Several reviewers specifically note the scheduling conflict between Fantasmic and the night parade, and multiple reviewers recommend staying in the park until close to catch both shows.


**Topic 5 — Family Milestones & Children's First Disney Visits** *(Prevalence: 12.1%)*

Top words: `year, kid, old, year old, loved, love, every, family, first, time`

This is the most prevalent topic for California and captures emotionally resonant reviews from parents bringing young children for the first time, or adults returning after many years away. Reviews frequently express a strong generational connection to the park. Sentiment is strongly positive and nostalgic. In the representative documents, reviewers describe 2- and 3-year-olds who haven't stopped talking about the trip since returning home, a teenager who requested a return trip as her high school graduation gift, parents who felt like children again themselves, and families who have made annual trips for six or seven consecutive years.

### Paris


**Topic 1 — Ticket Purchasing, Transport & Pre-Visit Planning** *(Prevalence: 8.5%)*

Top words: `ticket, pas, paris, fast, studio, fast pas, train, rer, online, book`

This topic groups reviews that advise on how to get to and into the park — booking tickets online, navigating the RER A train from central Paris, managing two-park passes, and using Fast Pass. Sentiment is practical and informative. In the representative documents, reviewers note that online tickets cost approximately €47 per person versus €90 at the gate, that the RER A takes about 1 hour 20 minutes from central Paris, and that Walt Disney Studios closes at 19:00 while the main Disneyland Park closes at 23:00. One reviewer details the disability pass process, which requires a doctor's letter and proof of DLA to obtain a Disney green pass from City Hall inside the park.



**Topic 2 — Overpriced Food, Drink & Unreliable Fastpass System** *(Prevalence: 12.2%)*

Top words: `food, queue, euro, expensive, drink, fast, hour, water, bring, snack`

Reviews in this topic share frustration with the cost of food and drink inside the park and the unreliability of the Fastpass system. Visitors frequently advise future visitors to bring their own food and fill up water bottles at the park's water fountains. Sentiment is strongly negative. In the representative documents, reviewers describe beer costing €8 per pint, kids' Coke at €3.75, Fastpass machines breaking down with queues of 100 people, and waiting 40 minutes at a so-called fast food counter. One reviewer explicitly tells future visitors to "save your money and take your kids to the local rec" rather than pay the park's food prices.



**Topic 3 — Staff Attitude, Ride Closures & Park Deterioration** *(Prevalence: 11.6%)*

Top words: `staff, would, closed, member, paris, year, could, rude, attraction, refurbishment`

This topic captures reviews dominated by disappointment with staff behaviour and a high frequency of attraction closures. Paris is repeatedly and unfavourably compared to US Disney parks in terms of service standards and park upkeep. Sentiment is strongly negative. In the representative documents, reviewers describe French cast members as surly and dismissive of English speakers, finding 15–20% of attractions closed for refurbishment during a single visit, peeling paint, cigarette smoke in non-smoking areas, and overflowing bins. One reviewer emailed Disney directly about their experience and received no reply, calling it unprofessional.


**Topic 4 — Queue Wait Times & Crowd Management for Families** *(Prevalence: 10.2%)*

Top words: `attraction, child, kid, queue, minute, waiting, hour, long, busy, summer`

Reviews in this topic focus on wait times and how they affect family visits — particularly parents with young children struggling with long queues during peak summer season. Sentiment ranges from resigned acceptance to outright frustration. In the representative documents, reviewers describe spending 8 hours in the park and completing only 3 rides, facing 90-minute queues for single attractions, and one reviewer who proposed implementing digital queue numbers rather than requiring visitors to stand in physical lines. Several reviewers specifically recommend visiting in off-peak months such as January, February, or November to avoid the worst crowds.


**Topic 5 — Paris vs US Disney Parks: Direct Comparisons** *(Prevalence: 8.9%)*

Top words: `paris, florida, world, orlando, magic, attraction, compare, castle, beautiful, visit`

This topic groups reviews that explicitly compare Disneyland Paris to Walt Disney World Orlando or Disneyland California. Sentiment is split — some visitors are pleasantly surprised by Paris; others use the comparison to discourage future visits. In the representative documents, reviewers praise the Paris castle's stained-glass windows and its hidden animatronic dragon beneath Sleeping Beauty's castle as superior to California. Others criticise Paris for outdated attractions and inferior cleanliness compared to US parks. One reviewer who has visited US Disney parks over 10 times describes Paris as the first time they genuinely felt the Disney magic.

### Hong Kong


**Topic 1 — Park Scale & Managing Visitor Expectations** *(Prevalence: 12.9%)*

Top words: `hong, kong, hong kong, smaller, small, visit, compare, tokyo, world, day`

This topic captures reviews where visitors explicitly reflect on Hong Kong Disneyland's smaller scale relative to other Disney parks worldwide. Reviewers are largely managing expectations for others rather than expressing strong emotion. Sentiment is measured and factual. In the representative documents, reviewers who have visited Tokyo, Orlando, Anaheim, and Paris all describe Hong Kong as the smallest of the Disney parks, note they were able to walk straight onto rides due to low crowds, and recommend the park primarily for families with young children or as a day trip. One reviewer states it should not be the top item on a Hong Kong itinerary but is worth fitting in if time allows.



**Topic 2 — Overpriced Food & Bag-Check Restrictions** *(Prevalence: 7.3%)*

Top words: `food, water, expensive, bring, drink, inside, bottle, price, outside, snack`

Reviews in this topic focus on the high cost of food and drink combined with bag checks at the entrance that prevent visitors bringing their own food inside. Sentiment is negative specifically around pricing, though overall park enjoyment is often still rated positively. In the representative documents, reviewers describe paying $5 AUD for a single bottle of water, being advised to bring their own water bottle to fill at the park's fountains, finding many food stalls closed and having to walk a long distance to find refreshments, and vegetarian visitors being frustrated by very limited food options. One reviewer describes Disney as having visitors "over a barrel" because bag checks prevent bringing outside food while internal options are expensive and limited.



**Topic 3 — Queue-Jumping & Crowd Behaviour Complaints** *(Prevalence: 14.4%)*

Top words: `staff, would, food, queue, small, well, attraction`

This is the most prevalent topic for Hong Kong and captures a recurring and specific complaint about crowd behaviour — particularly perceived queue-cutting and pushing by visitors from mainland China. While the top words are generic, the representative documents consistently and explicitly describe this pattern. Sentiment is strongly negative. In the representative documents, reviewers describe politely observing queue-jumping behaviour that would not be tolerated at Anaheim, staff appearing unable or unwilling to manage the situation, and the problem being worst during mainland China public holiday periods. One reviewer explicitly advises European visitors to go to Paris instead because of this issue.



**Topic 4 — Character Meet-and-Greet Queues & Photo Opportunities** *(Prevalence: 9.7%)*

Top words: `queue, ticket, photo, take, pas, character, arrive, early, long, wait`

Reviews in this topic focus on interactions with Disney characters — specifically the frustration of long queues for photo opportunities and limited character availability walking freely through the park. Sentiment is mixed: disappointment about character access is common, but the parade is consistently cited as a positive fallback. In the representative documents, reviewers describe joining a character photo queue only to abandon it after waiting too long with no visible end in sight, seeing characters only briefly during the parade rather than walking freely through the park, and confusion around third-party ticket purchases causing 30-minute delays at Guest Relations. One reviewer states that without characters walking freely around the park, Disneyland loses its sense of magic.



**Topic 5 — Family-Friendly Experience & Suitability for Young Kids** *(Prevalence: 14.4%)*

Top words: `kid, child, visit, adult, young, family, small, recommend, great, enjoy`

This topic captures broadly positive reviews from families who found Hong Kong Disneyland well-suited to their needs — particularly for younger children. Reviewers frequently position the park as ideal for a specific audience and offer practical recommendations. Sentiment is positive and practical. In the representative documents, reviewers describe riding every ride multiple times due to short queues on weekday visits, recommend visiting in winter for cooler and less crowded conditions, and explicitly position the park as ideal for children aged 3 to 9 or for families who have never visited a Disney park before. Several reviewers who have also visited larger Disney parks note that Hong Kong is the right size for a manageable one-day family outing.

## Topic Interpretations — Student B (Topics 6–10)

This section presents the interpretation of Topics 6–10 for each Disneyland location (California, Paris, Hong Kong) based on the NMF model results.

Each topic is interpreted using top words, representative meaning, and sentiment, following the same structure as Student A to ensure consistency across the analysis.

### California

**Topic 6 — Family Experiences & Kids Activities**

**Top words:** kid, fun, family, child, adult, age, food  

This topic captures reviews centered around family visits, particularly those involving children enjoying the park. Visitors frequently describe Disneyland as a space where both kids and adults can have fun together, emphasizing its appeal across age groups. Reviews often highlight how attractions are designed to be inclusive for families, with specific mentions of children being excited and entertained throughout the visit.  

Sentiment is strongly positive, as many reviews frame Disneyland as an ideal destination for family bonding and shared experiences.

**Topic 7 — First-Time Visits & Emotional Experiences**

**Top words:** year, old, daughter, son, first, love  

This topic reflects emotionally driven reviews, particularly those describing first-time visits to Disneyland. Many reviews focus on children experiencing the park for the first time, with parents highlighting the excitement and emotional significance of these moments. Visitors often describe these experiences as memorable milestones, reinforcing Disneyland’s reputation as a “once-in-a-lifetime” or highly meaningful destination.  

Sentiment is overwhelmingly positive and nostalgic, with strong emotional language centered on family memories and personal significance.

**Topic 8 — Staff & Service Quality**

**Top words:** member, cast, friendly, staff  

This topic highlights interactions with Disney employees (cast members), with reviews frequently emphasizing friendliness, professionalism, and helpfulness. Visitors often describe staff as approachable and attentive, contributing positively to the overall park experience.  

Sentiment is strongly positive, indicating that service quality is a key strength of the California Disneyland experience.

**Topic 9 — Shows, Parades & Night Entertainment**

**Top words:** parade, firework, show, night  

This topic focuses on Disneyland’s entertainment offerings, particularly nighttime shows, parades, and fireworks. Reviews consistently describe these events as highlights of the visit, with many visitors recommending staying late to experience them.  

Sentiment is overwhelmingly positive, with frequent use of strong descriptive language indicating that these performances play a central role in visitor satisfaction.

**Topic 10 — Overall Satisfaction & Repeat Visits**

**Top words:** love, always, going, visit  

This topic reflects overall satisfaction and loyalty toward Disneyland. Many reviews express a desire to return, with visitors describing the park as a place they “always” enjoy and plan to revisit.  

Sentiment is strongly positive, suggesting high levels of customer satisfaction and long-term emotional attachment to the park.

### Paris

**Topic 6 — Queue Times & Waiting Experience**

**Top words:** queue, hour, long, wait  

This topic captures visitor frustration with long wait times and crowded conditions. Reviews frequently mention spending significant time in queues, which negatively impacts the overall experience.  

Sentiment is predominantly negative, highlighting operational challenges related to crowd management.

**Topic 7 — Family Visits & Children Experiences**

**Top words:** year, old, daughter, child  

This topic focuses on family visits involving children, similar to California. However, the tone is more descriptive than emotional, with less emphasis on nostalgia and more on practical experiences.  

Sentiment is generally positive, reflecting the park’s appeal to families despite other challenges.

**Topic 8 — FastPass Usage & Planning Strategies**

**Top words:** fast pass, use, pass  

This topic highlights how visitors use the FastPass system to manage time and reduce waiting. Reviews often discuss strategies for navigating the park efficiently.  

Sentiment is mixed to neutral, as FastPass is seen as useful but not always reliable.

**Topic 9 — Shows & Entertainment**

**Top words:** show, parade, firework, amazing  

This topic reflects positive experiences related to shows and parades, similar to California. Visitors frequently describe these as standout moments of their trip.  

Sentiment is strongly positive, indicating that entertainment remains a major strength of the park.

**Topic 10 — Pricing & Accessibility**

**Top words:** ticket, train, buy, expensive  

This topic captures concerns related to ticket prices and transportation costs. Many reviews highlight the high cost of visiting Disneyland Paris and logistical considerations for getting there.  

Sentiment is largely negative, reflecting price sensitivity among visitors.

### Hong Kong

**Topic 6 — Family-Friendly Experience**

**Top words:** year, old, daughter, son  

This topic reflects family visits and shared experiences, particularly for visitors with young children. Reviews emphasize the suitability of the park for families.  

Sentiment is positive, reinforcing the park’s positioning as family-oriented.

**Topic 7 — Signature Attractions & Shows**

**Top words:** show, lion king, mickey  

This topic highlights specific attractions and performances, particularly well-known shows like The Lion King. Visitors often describe these as key highlights of their visit.  

Sentiment is strongly positive, indicating high satisfaction with entertainment offerings.

**Topic 8 — Park Size & Comparisons**

**Top words:** small, child, world, compared  

This topic reflects comparisons between Hong Kong Disneyland and other Disney parks. Many reviews describe the park as smaller, influencing visitor expectations.  

Sentiment is neutral to mixed, as smaller size is seen as both a limitation and an advantage (easier navigation).

**Topic 9 — Queue Times & Crowding**

**Top words:** queue, long, wait  

This topic captures waiting times and crowd-related frustrations. While present, these issues appear less severe compared to Paris.  

Sentiment is negative, though less intense than in other locations.

**Topic 10 — Transportation & Accessibility**

**Top words:** train, mtr, transport  

This topic focuses on transportation, particularly the convenience of public transit like the MTR. Reviews often highlight ease of access to the park.  

Sentiment is generally neutral to positive, with accessibility viewed as a practical advantage.

## Student B — Key Insights

Across all three locations, several consistent themes emerge, including family experiences, entertainment, and waiting times. However, there are notable differences:

- California emphasizes emotional connection and repeat visitation  
- Paris highlights operational challenges such as pricing and queues  
- Hong Kong reflects comparisons in scale and accessibility  

Overall, the NMF model successfully identifies meaningful patterns in visitor experiences, allowing for clear cross-location comparisons and actionable insights into customer satisfaction.